## Libraries import

In [17]:
import os
from torch.utils.data import Subset, DataLoader
from torchvision import transforms
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import copy
from datasets.celebA_dataset import CelebADataset
from torchvision.models import vit_b_16, ViT_B_16_Weights

## Dataset Class

## Data Loading & Transform

In [6]:
base_path = 'AdvCelebA'
label_file = os.path.join(base_path, 'attack_CelebA.txt')
image_dir = os.path.join(base_path, 'images')
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')

transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),       
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = CelebADataset(label_file, image_dir, partition_file, transform=transform)

train_dataset = Subset(dataset, dataset.train_indices)
val_dataset = Subset(dataset, dataset.val_indices)
test_dataset = Subset(dataset, dataset.test_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(val_dataset, batch_size=32)

## Model CNN

In [18]:
class ViTBinaryClassifier(nn.Module):
    def __init__(self, num_classes: int = 1):
        """
        Initializes the ViTBinaryClassifier.

        Args:
            num_classes (int): The number of output classes. For binary classification,
                               this should typically be 1 (for logits) or 2 (for probabilities
                               with CrossEntropyLoss). Here, we use 1 for a single logit output.
        """
        super(ViTBinaryClassifier, self).__init__()
        self.vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        for param in self.vit.parameters():
            param.requires_grad = False

        self.vit.heads.head = nn.Sequential(
            nn.Linear(self.vit.hidden_dim, 256), 
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass through the ViT model.

        Args:
            x (torch.Tensor): The input tensor, expected to be of shape (batch_size, 3, H, W).

        Returns:
            torch.Tensor: The output logits for binary classification.
        """
        return self.vit(x)

## Training Loop

In [19]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

In [12]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = ViTBinaryClassifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30
patience = 5
best_val_loss = float('inf')
early_stop_counter = 0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        print(f"🔁 No improvement in validation loss for {early_stop_counter} epochs.")

        if early_stop_counter >= patience:
            print(f"⏹️ Early stopping triggered at epoch {epoch+1}")
            break

model.load_state_dict(best_model_wts)

Epoch 1 - Training: 100%|██████████| 5072/5072 [1:43:32<00:00,  1.22s/it]


Epoch 1, Train Loss: 0.3587, Val Loss: 0.3405


Epoch 2 - Training: 100%|██████████| 5072/5072 [1:46:32<00:00,  1.26s/it]


Epoch 2, Train Loss: 0.3426, Val Loss: 0.3360


Epoch 3 - Training: 100%|██████████| 5072/5072 [1:55:21<00:00,  1.36s/it]


...

Epoch 4, Train Loss: 0.3347, Val Loss: 0.3303


Epoch 5 - Training: 100%|██████████| 5072/5072 [1:50:41<00:00,  1.31s/it]


Epoch 5, Train Loss: 0.3317, Val Loss: 0.3331
🔁 No improvement in validation loss for 1 epochs.


Epoch 6 - Training:   8%|▊         | 383/5072 [08:17<1:41:26,  1.30s/it]


KeyboardInterrupt: 

In [13]:
model.load_state_dict(best_model_wts)

<All keys matched successfully>

In [16]:
torch.save(model.state_dict(), 'models_bin/vit_binary_classifier.pth')

In [21]:
model = ViTBinaryClassifier().to(device)
model_path = "models_bin/vit_binary_classifier.pth"
model.load_state_dict(torch.load(model_path, map_location=device))

<All keys matched successfully>

## Validation Accuracy

In [22]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs).cpu().numpy() > 0.5
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {acc:.4f}")

Validation Accuracy: 0.8652
